# 04. Agent Monitoring: 평가와 무엇이 다르고, 무엇을 언제 경보하는가

03a와 03b에서는 **한 번 모은 Episode를 채점**했습니다. 이번 노트북은 질문이 바뀝니다. 운영이 이틀째 돌아가고 있을 때, **어제와 오늘이 달라졌다는 것을 누가 먼저 알아채는가**입니다. 평가는 "이 Episode는 PASS인가"를 묻고, 모니터링은 "지표가 어제와 같은 모양인가, 기준을 넘었는가"를 묻습니다.

## 수업 시간표

| 구간 | 내용 | 분 |
|---|---|---|
| 1 | 모니터링과 평가의 차이, 준비 | 5 |
| 2 | 모의 운영 트래픽 이틀치 만들기 (버전 라벨, release·environment 태그) | 8 |
| 3 | 지표 체계를 로컬 원자료로 계산하기 | 7 |
| 4 | Metrics API v2 질의, 원격과 로컬 비교, 집계 지연 | 8 |
| 5 | 입력 분포 변화(전체 변동 거리)와 사전 고정 경보 기준 | 8 |
| 6 | OTel GenAI 이름, 직접 해보기, 정리 | 9 |

## 이 노트북이 하는 일

- 설계 문서 9장의 지표(치명 위반률, 경고율, 평가 완료율, 지연 p50/p95, 성공 건당 비용, 도구 오류율, 입력 분포 변화)를 `eval_lab.monitoring.local_metrics`로 계산합니다.
- Langfuse 클라이언트가 있으면 같은 지표 일부를 Metrics API v2로 묻고 로컬 값과 나란히 놓습니다. 없으면 로컬 경로만 진행합니다. 둘 다 끝까지 돌아갑니다.
- 경보 기준 `ALERT_RULES`는 **결과를 보기 전에 코드에 고정된 상수**입니다. 2일차에 적용해 어떤 경보가 울리는지 봅니다.

## 호출 규모

이 노트북은 **유료 모델 호출이 없습니다.** 모의 운영 트래픽은 scripted 드라이버가 만들고(모델 0회), 판정은 규칙 기반 궤적 판정입니다. Langfuse 키가 있으면 trace를 남기고 Metrics API를 조회하지만, 이것은 모델 호출이 아닙니다.

## 먼저 알아 둘 것

- scripted 트래픽은 **모의 트래픽**입니다. 여기서 나온 수치를 LLM 성능으로 읽지 않습니다. 실수는 일부러 섞은 것이고, 목적은 "지표와 경보가 그 변화를 잡는가"를 보는 것입니다.
- 이 노트북이 확인하지 못한 것(Langfuse 서버의 실제 집계 지연, Metrics API 응답의 세부 값, OTel GenAI 규약의 안정화)은 해당 위치에서 **미확인**이라고 적습니다.

## 1. 모니터링은 평가와 무엇이 다른가

같은 Episode 기록을 보더라도 묻는 것이 다릅니다.

| | 평가(03a·03b) | 모니터링(이번 노트북) |
|---|---|---|
| 묻는 것 | 이 Episode가 성공인가 | 어제와 오늘의 **모양**이 같은가 |
| 단위 | Episode 하나, 또는 표본 | 날짜·버전·문형 같은 **그룹** |
| 기준 | 규칙과 판정자 | 사전에 고정한 **경보 기준** |
| 정답 | 없어도 되는 지표만 운영에서 사용 | 정답이 필요 없는 지표만 사용 |
| 실패 모양 | FAIL 한 건 | 비율이 기준을 넘음 |

모니터링 지표는 크게 세 갈래에서 옵니다.

1. **점수에서 오는 것**: 치명 위반률, 경고율, 평가 완료율. 03b의 궤적 판정 결과를 세어서 얻습니다.
2. **기록(Metrics API)에서 오는 것**: 지연 p50/p95, 비용. root observation의 latency와 비용을 묻습니다.
3. **메타데이터에서 오는 것**: 입력 분포 변화. 요청 유형(가족·문형) 비율이 날짜 사이에 얼마나 달라졌는지 봅니다.

한 가지가 더 중요합니다. 모니터링은 **"평가 불능"을 따로 세야** 합니다. 평가 완료율이 떨어졌는데 PASS 비율만 보면 "문제 없음"으로 읽힙니다. 03a·03b에서 본 INCOMPLETE, PROVISIONAL, NO_EVIDENCE를 이번에도 따로 봅니다.

In [ ]:
# [따라하기] 준비: 도구를 가져오고, 결과를 둘 폴더와 Langfuse 연결을 정한다
import json
import time
from dataclasses import FrozenInstanceError
from datetime import UTC, datetime

from eval_lab.monitoring import (
    ALERT_RULES,
    REMOTE_LAG_NOTE,
    ROOT_FILTER,
    compare_remote_local,
    evaluate_alerts,
    fetch_metrics,
    local_metrics,
    metrics_query,
    otel_names,
    total_variation_distance,
)
from eval_lab.notebook import announce, langfuse_or_none, new_run_dir, show
from eval_lab.traffic import (
    DAY_MIX,
    SHIFTED_FAMILIES,
    make_mistake_driver,
    make_traffic,
    run_traffic,
)

# 1단계: 이 노트북이 만드는 파일은 모두 이 폴더 안에만 쓴다.
run_dir = new_run_dir("04")
print("결과 폴더:", run_dir.name)

# 2단계: Langfuse 키가 없으면 None이 돌아온다. 이후 셀은 None이어도 끝까지 돈다.
langfuse = langfuse_or_none()
print("Langfuse 연결:", "있음" if langfuse is not None else "없음(로컬 경로만 사용)")

In [ ]:
# 스위치: 기본은 모두 꺼 둔다. 켜면 직접 해보기의 큰 실행과 선택 셀이 추가로 돈다.
RUN_EXERCISE = False
RUN_OPTIONAL = False

## 2. 모의 운영 트래픽 이틀치

`make_traffic(day, n, seed=...)`은 날짜별 분포(`DAY_MIX`)에 따라 요청 n건을 만듭니다. 같은 `(day, n, seed)`는 같은 결과입니다.

- **1일차**는 정상 요청(`incident-normal`)이 많은 분포입니다.
- **2일차**는 환불 TIMEOUT 두 가지, 결제 장애, 다른 고객 주문이 늘어난 분포입니다. 이 가족들을 `SHIFTED_FAMILIES`가 모아 둡니다.

`run_traffic`에는 세 가지 라벨을 붙입니다.

| 라벨 | 인자 | Langfuse에서의 역할 |
|---|---|---|
| 정책 버전 | `version="r0"` | observation의 version. "어느 정책이 낸 trace인가"를 가른다 |
| release | `release="baseline"` | 시스템 지침의 종류(baseline 또는 candidate) |
| environment | `environment_tag` | 운영과 실험을 가르는 환경 이름. Metrics API 필터에 쓴다 |

2일차에는 scripted 드라이버가 **실수를 일부러 섞습니다**(`make_mistake_driver`). 승인 전에 쓰기, 기록 없이 완료라고 말하기 같은 실수가 실제 도구 호출로 일어나므로 궤적 규칙이 그것을 잡습니다. 1일차에는 실수를 섞지 않습니다.

**먼저 예측하세요.** 2일차의 입력 분포 변화와 치명 위반률 중 어느 쪽이 먼저 눈에 띄는 변화일까요? 둘 다 변할까요, 하나만 변할까요? 아래 셀을 돌리기 전에 한 줄로 적어 두세요.

In [ ]:
# [따라하기] 모의 운영 트래픽 이틀치를 만들고 실행한다
announce("모의 운영 트래픽 두 날(scripted, 모델 호출 없음)", model_calls=0)

# 1단계: 날짜별 요청을 만든다. seed가 같으면 같은 요청이 나온다.
ENV = "day05_nb04"
VERSION = "r0"
DAY_SIZE = 30
SEED = 7
items = {day: make_traffic(day, DAY_SIZE, seed=SEED) for day in (1, 2)}

# 2단계: 1일차는 정상 흐름으로 돌린다. 시각을 재 두는 것은 4절에서 Metrics API 조회 구간으로 쓰기 위해서다.
started = datetime.now(UTC)
records_day1 = run_traffic(items[1], langfuse=langfuse, release="baseline", version=VERSION, environment_tag=ENV)
day1_end = datetime.now(UTC)

# 3단계: 2일차는 실수를 섞는 드라이버로 돌린다.
MISTAKE_RATES = {"write_before_approval": 0.2, "claim_without_record": 0.1}
mistake_driver = make_mistake_driver(MISTAKE_RATES, seed=11)
records_day2 = run_traffic(items[2], driver=mistake_driver, langfuse=langfuse, release="baseline",
                           version=VERSION, environment_tag=ENV)

# 4단계: Langfuse는 기록을 모아서 보낸다. 조회 전에 flush로 내보낸다.
if langfuse is not None:
    langfuse.flush()
day2_end = datetime.now(UTC)

records = records_day1 + records_day2
show([
    {"일차": day, "Episode 수": len(rows),
     "실수를 시도한 Episode": sum(1 for row in rows if row["injected_mistakes"]),
     "끝나지 못한 Episode": sum(1 for row in rows if row["execution_status"] != "completed")}
    for day, rows in ((1, records_day1), (2, records_day2))
], title="이틀치 모의 트래픽")

**관찰할 것.** 두 날 모두 Episode가 30건이고, 2일차에만 "실수를 시도한 Episode"가 있습니다. 실수는 `injected_mistakes`에 이름이 남습니다. 모니터링 지표는 이 실수 목록을 **보지 않고**, 기록된 도구 호출 궤적만으로 판정합니다. 실수 목록은 지표가 맞게 잡았는지 확인하는 용도입니다.

Langfuse 연결이 있으면 같은 Episode가 trace로도 남습니다. 연결이 없으면 `records`(로컬 원자료)만 남고, 이후 계산은 모두 이 `records`에서 합니다. **지표의 원자료는 언제나 로컬에 있다**는 점이 키가 없을 때의 대체 경로가 됩니다.

## 3. 지표 체계를 로컬 원자료로 계산하기

`local_metrics(records, group_by="day")`는 기록을 날짜별로 나눠 설계 문서 9장의 지표를 모두 계산합니다. 핵심 규칙 세 가지입니다.

- **평가 완료율** = (PASS + FAIL) / 전체. INCOMPLETE, PROVISIONAL, NO_EVIDENCE는 따로 비율을 냅니다. 끝나지 못한 Episode를 분모에서 빼면 완료율이 거짓으로 높아집니다.
- **치명 위반률**의 분모는 판정 완료 Episode(PASS + FAIL)입니다. 판정 못 한 것은 위반으로도, 정상으로도 세지 않습니다.
- **알 수 없는 값은 0이 아니라 `None`**입니다. 예를 들어 비용을 모르는 기록이 하나라도 있으면 성공 건당 비용은 `None`입니다.

의미 판단 실패율과 Judge 넘김 비율은 `semantic_results`를 함께 넘길 때만 붙습니다. 그 값은 Jev·Judge 호출 결과가 있어야 하므로(03a에서 다룸) 이 노트북에서는 계산하지 않습니다.

In [ ]:
# [따라하기] 날짜별 지표를 계산해 표로 본다
metrics = local_metrics(records, group_by="day")

SHOWN = ["episode_count", "fatal_rate", "evaluation_completeness", "incomplete_share", "provisional_share",
         "no_evidence_share", "latency_p50", "latency_p95", "cost_per_success", "tool_error_rate"]


def metric_table(by_group, names, suffix):
    """그룹별 지표 dict를 지표 이름이 행인 표로 바꾼다. 실수는 소수 넷째 자리까지만 보인다."""
    rows = []
    for name in names:
        row = {"지표": name}
        for group, values in by_group.items():
            value = values[name]
            row[f"{group}{suffix}"] = round(value, 4) if isinstance(value, float) else value
        rows.append(row)
    return rows


show(metric_table(metrics, SHOWN, "일차"), title="날짜별 지표(로컬 원자료)")
show([{"일차": day, **values["status_counts"]} for day, values in metrics.items()], title="판정 상태별 Episode 수")
show([{"일차": day, "규칙": rule, "경고율": round(rate, 4)}
      for day, values in metrics.items() for rule, rate in values["warn_rate"].items()], title="규칙별 경고율")

# 지표 dict를 결과 폴더에 남긴다. 키나 비밀 값은 들어 있지 않다.
(run_dir / "metrics_by_day.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

**관찰할 것.** 두 열을 위에서 아래로 읽으세요.

- `fatal_rate`: 1일차는 0이고 2일차는 0보다 큽니다. 섞어 넣은 실수가 궤적 규칙에 걸린 것입니다.
- `tool_error_rate`: 2일차가 더 높습니다. 결제 장애와 TIMEOUT 가족이 늘어 서비스가 오류 결과를 더 자주 돌려준 것입니다. 이것은 에이전트의 실수가 아니라 **입력이 바뀐 결과**입니다. 같은 지표라도 원인이 다를 수 있다는 점을 기억해 두세요.
- `cost_per_success`: scripted 트래픽은 모델을 부르지 않으므로 비용이 0입니다. 실제 모델이 붙은 트래픽에서만 의미가 있고, 그때도 실패와 재시도 비용을 빼지 않습니다.
- `latency_p50`·`latency_p95`는 이 컴퓨터에서 scripted 실행에 걸린 시간입니다. 실행할 때마다 값이 조금 달라집니다. 이 값으로 모델 지연을 말할 수 없습니다.

In [ ]:
# [바꿔보기] 묶는 기준을 날짜에서 문형으로 바꾼다. 같은 기록이라도 그룹을 바꾸면 다른 질문에 답한다
by_style = local_metrics(records, group_by="style")
show(metric_table(by_style, ["episode_count", "fatal_rate", "evaluation_completeness", "tool_error_rate"], ""),
     title="문형별 지표(두 날을 합친 기록)")

**관찰할 것.** 문형(정중, 짧게, 오타)마다 치명 위반률이 조금씩 다르게 보이지만, 문형별 그룹은 각각 스무 건 안팎이라 한두 건 차이로 비율이 크게 움직입니다. 이 차이는 문형의 영향이 아니라 **드라이버가 섞은 실수가 어느 요청에 걸렸는가**에서 온 우연으로 읽는 편이 안전합니다. 모니터링에서 "어떤 축으로 쪼갰을 때 차이가 나는가"가 원인 추적의 첫걸음입니다. 다만 그룹이 작으면(문형마다 십수 건) 비율이 크게 흔들리므로, 차이를 곧바로 원인으로 읽지 않습니다.

In [ ]:
# [바꿔보기] 2일차에서 실수를 모두 끈 대조 실행. 입력 분포는 그대로 두고 실수만 없앤다
announce("실수를 끈 대조 실행(scripted, 모델 호출 없음)", model_calls=0)
control_driver = make_mistake_driver({name: 0.0 for name in MISTAKE_RATES}, seed=11)
control_records = run_traffic(items[2], driver=control_driver, release="baseline", version=VERSION,
                              environment_tag=f"{ENV}_control")
control_metrics = local_metrics(control_records, group_by="day")
show([{"실행": "2일차(실수 포함)", "fatal_rate": metrics[2]["fatal_rate"], "tool_error_rate": round(metrics[2]["tool_error_rate"], 4)},
      {"실행": "2일차 대조(실수 없음)", "fatal_rate": control_metrics[2]["fatal_rate"],
       "tool_error_rate": round(control_metrics[2]["tool_error_rate"], 4)}], title="실수 유무에 따른 차이")

**관찰할 것.** 실수를 끄면 `fatal_rate`가 0으로 돌아가지만 `tool_error_rate`는 거의 같습니다. 도구 오류는 입력(결제 장애, TIMEOUT 가족)에서 오고, 치명 위반은 에이전트 행동에서 옵니다. 두 지표가 **서로 다른 원인에 반응한다**는 것이 지표를 여러 개 두는 이유입니다. 대조 실행은 Langfuse에 남기지 않았습니다(`langfuse`를 넘기지 않음). 다음 절의 원격 조회에 섞이지 않게 하기 위해서입니다.

## 4. Metrics API v2: 원격 지표와 로컬 지표

로컬 계산은 키 없이 되지만, 운영에서는 trace가 이미 Langfuse에 쌓여 있으므로 **서버가 집계해 주는 값**을 묻는 편이 빠릅니다. `metrics_query`가 질의 dict를 만들고 `fetch_metrics`가 보냅니다.

- **view는 `observations`입니다.** v2에는 `traces` view가 없습니다. trace 단위 지연·비용은 **root observation**(부모가 없는 observation)으로 묻습니다. `ROOT_FILTER`가 그 필터이고, `filters=None`이면 observations view에 자동으로 들어갑니다.
- 측정값은 `(measure, aggregation)` 쌍입니다. 예: `("latency", "p95")`, `("count", "count")`.
- 시간 구간(`from_ts`, `to_ts`)을 반드시 줍니다. 날짜별 비교는 구간을 나눠 같은 질의를 두 번 보내는 방식입니다.
- environment 필터로 이 노트북이 만든 trace만 고릅니다. 같은 프로젝트의 다른 trace가 섞이면 값이 달라집니다.

**먼저 예측하세요.** 로컬 `episode_count`와 원격 `count_count`가 같을까요? `latency_p95`는 같을까요, 다를까요? 다르다면 이유를 두 가지 적어 보세요.

In [ ]:
# [따라하기] 날짜 구간마다 같은 질의를 보내 root observation의 개수와 p95 지연을 묻는다
def remote_day(client, start, end):
    """한 구간의 root observation 개수와 p95 지연(ms)을 Metrics API v2로 묻는다."""
    # 1단계: root observation과 environment로 좁힌 질의를 만든다. 열 이름은 count_count, p95_latency로 돌아온다.
    query = metrics_query(
        metrics=[("count", "count"), ("latency", "p95")],
        filters=[dict(ROOT_FILTER), {"column": "environment", "operator": "=", "value": ENV, "type": "string"}],
        from_ts=start, to_ts=end)
    # 2단계: 행이 없으면 값을 모르는 것이다. 0으로 채우지 않는다.
    rows = fetch_metrics(client, query)
    row = rows[0] if rows else {}
    return {"episode_count": row.get("count_count"), "latency_p95": row.get("p95_latency")}


if langfuse is None:
    remote_by_day = {}
    print("Langfuse 연결이 없어 원격 조회를 건너뜁니다. 지표는 로컬 값만 있습니다.")
else:
    windows = {1: (started, day1_end), 2: (day1_end, day2_end)}
    remote_by_day = {day: remote_day(langfuse, *window) for day, window in windows.items()}
    rows = []
    for day, remote in remote_by_day.items():
        local = {name: metrics[day][name] for name in ("episode_count", "latency_p95")}
        rows += [{"일차": day, **row} for row in compare_remote_local(remote, local)["rows"]]
    show(rows, title="원격(Langfuse)과 로컬 비교: difference = 원격 - 로컬")
    print(REMOTE_LAG_NOTE)

**관찰할 것.** 연결이 있었다면 표의 `difference` 열을 보세요.

- `episode_count`가 같으면 root observation 하나가 Episode 하나라는 약속이 지켜진 것입니다. 다르면 아직 집계되지 않은 trace가 있거나 일부만 반영된 것입니다. **집계는 비동기**라 방금 기록한 trace는 늦게 나타날 수 있습니다. 이 지연이 실제 서버에서 얼마나 되는지는 이 노트북에서 확인하지 못했습니다(**미확인**).
- `latency_p95`는 거의 확실히 다릅니다. 로컬은 Episode 전체 시간이고 원격은 root observation의 latency라 **측정 구간의 정의가 다릅니다.** 두 값이 같기를 기대하지 말고, 같은 쪽 값끼리 날짜를 비교하세요.
- 연결이 없었다면 "건너뜁니다" 한 줄만 나옵니다. 이것은 실패가 아니라 로컬 전용 경로입니다.

원격 숫자는 두 가지를 확인한 뒤에 믿습니다: 조회 구간이 Episode 시작 시각을 덮는지, environment 필터가 맞는지.

In [ ]:
# [깨뜨리기] v2에는 traces view가 없다. 예전 습관대로 traces로 물으면 질의를 만들 때 막힌다
try:
    metrics_query(view="traces", metrics=[("count", "count")], from_ts=started, to_ts=day2_end)
except ValueError as error:
    print("막힌 이유:", error)

**관찰할 것.** 질의를 서버로 보내기 **전에** `ValueError`가 납니다. 서버가 400을 돌려주기를 기다리지 않고 만드는 쪽에서 막은 것입니다. trace 단위 값이 필요하면 view는 `observations`로 두고 root observation 필터를 걸어야 합니다.

In [ ]:
# [깨뜨리기] 조회 구간이 Episode를 덮지 못하면 값이 없다. 구간을 일부러 미래로 옮겨 본다
if langfuse is None:
    print("Langfuse 연결이 없어 건너뜁니다.")
else:
    future = datetime.now(UTC)
    empty = remote_day(langfuse, future, future)
    print("미래 구간의 원격 값:", empty)
    print("로컬에는 Episode가", metrics[2]["episode_count"], "건이 있는데 원격은 위와 같다 -> 구간이 틀린 것이지 Episode가 사라진 것이 아니다")

**관찰할 것.** 구간이 틀리면 원격 개수가 0이거나 `None`으로 나옵니다. 이것을 "오늘 트래픽이 없다"로 읽으면 안 됩니다. 로컬 원자료와 나란히 놓는 이유가 이것입니다: **한쪽이 0인데 다른 쪽이 0이 아니면 조회가 틀렸거나 집계가 늦은 것**이고, 둘 다 0이어야 비로소 "없다"고 말할 수 있습니다.

## 5. 입력 분포 변화와 경보 기준

### 입력 분포 변화: 전체 변동 거리

모델도 정책도 그대로인데 지표가 변하는 가장 흔한 이유는 **입력이 바뀌었기 때문**입니다. 두 날의 요청 유형 비율을 `p`, `q`라 하면 전체 변동 거리(total variation distance)는 `0.5 * sum(|p - q|)`입니다. 0이면 같은 분포이고 1이면 겹치는 요청이 없습니다.

**먼저 예측하세요.** 가족(`family_id`) 분포와 문형(`style`) 분포 중 2일차에 더 크게 변한 쪽은 어디일까요? 거리의 크기를 대략 적어 보세요.

In [ ]:
# [따라하기] 가족과 문형 분포의 날짜 사이 거리를 계산한다
mix1, mix2 = metrics[1]["input_mix"], metrics[2]["input_mix"]
distances = {name: total_variation_distance(mix1[name], mix2[name]) for name in ("family_id", "style")}

# 설계한 분포(DAY_MIX)만으로 계산한 거리. 추첨 오차가 없는 기준값이다.
design = {day: {family: weight / sum(mix.values()) for family, weight in mix.items()} for day, mix in DAY_MIX.items()}
design_distance = total_variation_distance(design[1], design[2])

show([{"축": name, "관측한 거리": round(value, 3)} for name, value in distances.items()]
     + [{"축": "family_id(설계 분포)", "관측한 거리": round(design_distance, 3)}], title="입력 분포의 전체 변동 거리")

# 어떤 가족이 늘었는지 본다.
growth = sorted(((family, mix2["family_id"].get(family, 0.0) - mix1["family_id"].get(family, 0.0))
                 for family in set(mix1["family_id"]) | set(mix2["family_id"])), key=lambda pair: -pair[1])
show([{"가족": family, "비율 변화(2일차 - 1일차)": round(change, 3), "2일차에 늘기로 한 가족": family in SHIFTED_FAMILIES}
      for family, change in growth], title="가족별 비율 변화")

**관찰할 것.** 가족 분포의 거리는 크고 문형 분포의 거리는 작습니다. 문형은 두 날 모두 같은 가중치로 뽑았기 때문입니다. 가장 많이 늘어난 가족은 `SHIFTED_FAMILIES`에 든 것들입니다. 관측한 거리는 30건의 추첨이라 설계 값과 정확히 같지 않습니다. 표본이 작을수록 **분포 변화 경보는 오탐도 낸다**는 점을 기억하세요.

분포가 바뀌었다는 경보는 "에이전트가 나빠졌다"가 아니라 "이제부터 같은 기준으로 비교하면 안 될 수 있다"는 신호입니다.

### 경보 기준: 결과를 보기 전에 고정한 상수

`ALERT_RULES`는 `eval_lab/monitoring.py`에 상수로 적혀 있고, 이 노트북을 쓰기 전에 정해졌습니다. 기준을 결과를 본 뒤에 맞추면 경보는 **이미 일어난 일을 설명하는 도구**가 되어 쓸모가 없어집니다.

In [ ]:
# [따라하기] 고정된 경보 기준을 읽고, 2일차를 1일차(기준 그룹)에 견줘 평가한다
show([{"규칙": rule.rule_id, "지표": rule.metric, "기준": rule.threshold, "설명": rule.description}
      for rule in ALERT_RULES], title="사전 고정 경보 기준")

alerts = evaluate_alerts(metrics[2], metrics[1])
show([{"규칙": alert.rule, "관측값": round(alert.value, 4), "기준": round(alert.threshold, 4), "메시지": alert.message}
      for alert in alerts] or [{"규칙": "(울린 경보 없음)"}], title="2일차 경보")

baseline_alerts = evaluate_alerts(metrics[1], metrics[1])
print("1일차를 자기 자신에 견준 경보:", [alert.rule for alert in baseline_alerts] or "없음")

**관찰할 것.** 2일차에는 치명 위반률 경보와 입력 분포 경보가 울립니다. 평가 완료율 경보는 울리지 않습니다(이 트래픽의 Episode는 모두 끝까지 가서 판정이 났습니다). p95 지연 경보는 이 컴퓨터의 실행 시간에 달려 있어 울릴 수도, 안 울릴 수도 있습니다. 1일차를 자기 자신에 견주면 기준 그룹 대비 규칙(p95, 분포)은 울릴 수 없고, 치명 위반이 없으니 나머지도 조용합니다.

경보가 **두 개** 울렸다는 점이 중요합니다. 하나는 에이전트 행동(치명 위반)을, 다른 하나는 입력(분포)을 가리킵니다. 위의 대조 실행이 이 구분을 뒷받침합니다.

In [ ]:
# [바꿔보기] 실수를 끈 대조 실행에 같은 경보 기준을 적용한다. 기준은 그대로 두고 데이터만 바꾼다
control_alerts = evaluate_alerts(control_metrics[2], metrics[1])
show([{"실행": "2일차", "울린 경보": ", ".join(alert.rule for alert in alerts) or "없음"},
      {"실행": "2일차 대조(실수 없음)", "울린 경보": ", ".join(alert.rule for alert in control_alerts) or "없음"}],
     title="같은 기준, 다른 데이터")

**관찰할 것.** 대조 실행에서는 치명 위반률 경보가 사라지고 입력 분포 경보만 남습니다(지연 경보는 실행 시간에 달려 있습니다). 에이전트는 같고 입력만 달라졌을 때 "행동 경보는 조용하고 분포 경보는 운다"는 모양이 나옵니다. 이 모양을 보면 "입력이 바뀌었으니 Golden에 새 시나리오를 추가할지 검토하자"로 이어지고, 05에서 다루는 지속 운영 루프가 이 연결을 맡습니다.

In [ ]:
# [깨뜨리기] 계산하지 못한 지표(None)를 0이나 정상으로 읽으면 경보가 조용히 사라진다
unknown = {**metrics[2], "fatal_rate": None, "evaluation_completeness": None}
quiet = evaluate_alerts(unknown, metrics[1])
blind = [name for name in ("fatal_rate", "evaluation_completeness") if unknown[name] is None]

print("값을 모르는 지표:", blind)
print("그때 울린 경보:", [alert.rule for alert in quiet])
print("원래 울렸던 경보:", [alert.rule for alert in alerts])
print("사라진 경보:", sorted({alert.rule for alert in alerts} - {alert.rule for alert in quiet}))

**관찰할 것.** 치명 위반률을 계산하지 못하면 그 경보는 **울리지 않습니다.** `evaluate_alerts`는 `None`을 "정상"으로 해석하지 않고 "이 규칙은 판단 불가"로 건너뜁니다. 조용한 화면이 안전을 뜻하지 않습니다. 운영에서는 경보 목록과 함께 **값을 모르는 지표의 목록**도 반드시 보여 줘야 합니다. 이 셀의 `blind`가 그 목록입니다.

In [ ]:
# [깨뜨리기] 경보 기준을 결과를 본 뒤에 바꾸려 하면 막힌다
rule = ALERT_RULES[0]
try:
    rule.threshold = 0.5
except FrozenInstanceError as error:
    print("규칙 객체는 고쳐지지 않는다:", type(error).__name__)
print("규칙", rule.rule_id, "의 기준은 그대로:", rule.threshold)

**관찰할 것.** `AlertRule`은 값을 바꿀 수 없는 객체라 규칙 한 줄을 고치려는 시도가 막힙니다. 다만 이것은 **관례를 돕는 장치**이지 철벽이 아닙니다. 모듈의 상수를 코드에서 직접 덮어쓰면 평가가 바뀔 수 있습니다. 기준을 바꾸는 올바른 방법은 `monitoring.py`의 상수를 고치고 **변경 이력(커밋)에 남기는 것**입니다. 이 노트북에서는 결과를 본 뒤에 기준을 움직이지 않습니다.

## 6. OTel GenAI 이름

기록의 이름을 공급자마다 달리 부르면 도구를 바꿀 때마다 대시보드를 새로 만들어야 합니다. OpenTelemetry의 GenAI 규약은 **공급자에 묶이지 않는 이름**을 정합니다. 에이전트 실행은 `invoke_agent`, 도구 호출은 `execute_tool`, 평가 결과는 `gen_ai.evaluation.*` 속성으로 적습니다. `otel_names()`는 설치된 semconv 패키지의 상수에서 이 이름을 읽어 옵니다.

주의할 점이 있습니다. 설치된 패키지는 이 규약이 `semantic-conventions-genai` 저장소로 분리되었다고 표시합니다. 규약이 **안정(stable) 상태인지는 이 노트북이 1차 출처로 확인하지 못했습니다(미확인).** 이름을 소개할 뿐 "이 이름이 고정이다"라고 말하지 않습니다.

In [ ]:
# [따라하기] 설치된 semconv에서 OTel GenAI 이름을 읽어 온다
names = otel_names()
show([{"구분": "operation 속성", "이름": names["operation_attribute"]}]
     + [{"구분": f"operation 값({key})", "이름": value} for key, value in names["operation"].items()]
     + [{"구분": f"evaluation({key})", "이름": value} for key, value in names["evaluation"].items()], title="OTel GenAI 이름")
print(names["note"])

**관찰할 것.** 에이전트 실행과 도구 호출에 붙이는 이름이 짧고 공급자 이름이 없습니다. 우리가 Langfuse에 남기는 root observation이 `invoke_agent`, 도구 호출 observation이 `execute_tool`에 해당한다고 대응시킬 수 있지만, **현재 `eval_lab`은 이 이름을 기록에 붙이지 않습니다.** 대응은 개념 소개이고 구현이 아닙니다. 마지막 줄의 note가 안정화 여부를 확인하지 못했다고 말하는 것을 그대로 읽으세요.

## 직접 해보기

세 가지를 직접 해 봅니다. 기본값으로 바로 돌아갑니다. `RUN_EXERCISE = True`로 바꾸면 해보기 1의 Episode 수가 늘고, `RUN_OPTIONAL = True`로 바꾸면 해보기 4가 원격을 다시 조회합니다.

### 해보기 1

자기 트래픽을 만들어 같은 경보 기준에 적용합니다. `my_seed`, `my_rates`를 바꾸고, 어떤 경보가 울릴지 먼저 예상한 뒤 확인하세요. 기준(`ALERT_RULES`)은 건드리지 않습니다.

In [ ]:
# [내 것 만들기] 내 seed와 실수 확률로 2일차 모양의 트래픽을 만들어 경보를 확인한다
announce("내 트래픽(scripted, 모델 호출 없음)", model_calls=0)

# 1단계: 바꿔 볼 값. 스위치를 켜면 Episode 수가 늘어난다.
my_seed = 21
my_size = 60 if RUN_EXERCISE else 20
my_rates = {"write_before_approval": 0.1, "blind_retry": 0.1}

# 2단계: 2일차 분포로 요청을 만들고, 내 실수 확률로 실행한다.
my_records = run_traffic(make_traffic(2, my_size, seed=my_seed), driver=make_mistake_driver(my_rates, seed=my_seed),
                         release="baseline", version=VERSION, environment_tag=f"{ENV}_mine")
my_metrics = local_metrics(my_records, group_by="day")

# 3단계: 1일차를 기준으로 경보를 평가하고 결과를 폴더에 남긴다.
my_alerts = evaluate_alerts(my_metrics[2], metrics[1])
show(metric_table(my_metrics, ["episode_count", "fatal_rate", "evaluation_completeness", "tool_error_rate"], "일차(내 트래픽)"),
     title="내 트래픽 지표")
print("울린 경보:", [alert.rule for alert in my_alerts] or "없음")
(run_dir / "my_traffic_alerts.json").write_text(
    json.dumps({"seed": my_seed, "size": my_size, "rates": my_rates, "alerts": [alert.rule for alert in my_alerts]},
               ensure_ascii=False, indent=2), encoding="utf-8")

### 해보기 2

경보가 울렸는지가 아니라 **기준에서 얼마나 떨어져 있는지**를 봅니다. 여유가 작은 규칙은 다음 날 울릴 가능성이 높습니다. 기준은 읽기만 하고 바꾸지 않습니다.

In [ ]:
# [바꿔보기] 2일차 값이 각 기준에서 얼마나 떨어져 있는지 여유(margin)를 본다
current, baseline = metrics[2], metrics[1]
observed = {
    "fatal_rate": current["fatal_rate"],
    "evaluation_completeness": current["evaluation_completeness"],
    "latency_p95": current["latency_p95"] / baseline["latency_p95"] if baseline["latency_p95"] else None,
    "input_mix": max(distances.values()),
}
rows = []
for rule in ALERT_RULES:
    value = observed[rule.metric]
    # 완료율은 밑돌면 울리고, 나머지는 넘으면 울린다.
    margin = None if value is None else (value - rule.threshold if rule.metric == "evaluation_completeness"
                                          else rule.threshold - value)
    rows.append({"규칙": rule.rule_id, "관측값": None if value is None else round(value, 4),
                 "기준": rule.threshold, "여유(음수면 경보)": None if margin is None else round(margin, 4)})
show(rows, title="기준까지의 여유")

### 해보기 3

오늘 본 것을 **모니터링 일지**로 남깁니다. 일지에는 기준(결과를 보기 전의 값), 관측값, 울린 경보, 값을 모르는 지표를 함께 적습니다. 일지가 있어야 나중에 "그때 기준이 무엇이었나"를 되짚을 수 있습니다.

In [ ]:
# [내 것 만들기] 오늘의 모니터링 일지를 만든다
log = {
    "rules": [{"rule": rule.rule_id, "threshold": rule.threshold} for rule in ALERT_RULES],
    "observed": {name: (round(value, 4) if isinstance(value, float) else value) for name, value in observed.items()},
    "alerts": [alert.rule for alert in alerts],
    "unknown_metrics": [name for name, value in metrics[2].items() if value is None],
    "remote": "있음" if remote_by_day else "없음(로컬 전용)",
    "note": "모의 트래픽 기준이며 LLM 성능이 아니다. 원격 집계 지연은 확인하지 못했다.",
}
(run_dir / "monitoring_log.json").write_text(json.dumps(log, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(log, ensure_ascii=False, indent=2))

### 해보기 4 (선택)

`RUN_OPTIONAL = True`일 때만 돕니다. 원격 조회를 몇 초 뒤에 한 번 더 보내 값이 달라지는지 봅니다. 연결이 없으면 안내만 합니다.

In [ ]:
# [바꿔보기] 몇 초 뒤에 원격을 다시 조회해 집계 값이 달라지는지 본다
if not RUN_OPTIONAL:
    print("RUN_OPTIONAL = False라 건너뜁니다.")
elif langfuse is None:
    print("Langfuse 연결이 없어 건너뜁니다.")
else:
    time.sleep(3)
    again = remote_day(langfuse, day1_end, day2_end)
    print("처음 조회:", remote_by_day[2])
    print("다시 조회:", again)

## 확인 질문

1. 모니터링과 평가는 같은 Episode 기록에 무엇을 다르게 묻나요? 각각 한 문장으로 답해 보세요.
2. 평가 완료율의 분모를 "판정이 난 Episode"로 바꾸면 어떤 문제가 생기나요?
3. 원격 `latency_p95`와 로컬 `latency_p95`가 다른 이유를 두 가지 말해 보세요.
4. Metrics API v2에서 trace 단위 지연을 어떻게 묻나요? traces view로 물으면 어떻게 되나요?
5. 입력 분포 경보와 치명 위반률 경보가 함께 울렸을 때, 두 원인을 어떻게 가르나요?
6. 경보 기준을 2일차 결과를 본 뒤에 정하면 무엇이 잘못되나요?
7. 지표 값이 `None`일 때 경보가 울리지 않는 것을 어떻게 안전하게 다루나요?

## 정리

- 모니터링은 **그룹(날짜·버전·문형)의 모양**을 보고, 평가는 Episode의 성공을 봅니다. 평가 불능(INCOMPLETE, PROVISIONAL, NO_EVIDENCE)을 따로 세야 평가 완료율이 거짓말하지 않습니다.
- 지표의 원자료는 **로컬 기록**에 있고, Metrics API v2는 같은 값을 서버가 집계해 줍니다. view는 `observations`이고 traces view는 없습니다. trace 단위는 root observation으로 묻습니다. 두 값은 정의와 집계 시점이 달라 차이가 나며, 그 차이를 숨기지 않습니다.
- 입력 분포 변화는 전체 변동 거리로 재고, 경보 기준(`ALERT_RULES`)은 **결과를 보기 전에** 코드에 고정합니다. 행동 경보와 분포 경보를 가르면 원인을 구분할 수 있습니다.
- 값을 모르는 지표는 0이 아니라 `None`이고, `None`이면 경보가 조용합니다. 경보 목록과 함께 **모르는 지표 목록**을 봅니다.

**미확인으로 남은 것.** Langfuse 서버의 실제 집계 지연과 Metrics API 응답의 세부 값, 지연 값의 단위와 root observation latency의 정확한 정의, 2026-11-16 이후 Langfuse Cloud 동작, OTel GenAI 규약의 안정화 여부.

다음 노트북 05는 이 경보와 03a의 표본 선정을 **지속 운영 루프**로 묶습니다.